In [ ]:
import glob
import importlib
import os
import shutil
import subprocess
import sys
from pathlib import Path

# Configure environment variables for offline vLLM serving and LiteLLM routing
os.environ['LITELLM_LOCAL_MODEL_COST_MAP'] = 'True'
os.environ['TRANSFORMERS_NO_TF'] = '1'
os.environ['VLLM_WORKER_MULTIPROC_METHOD'] = 'spawn'
os.environ['VLLM_MEMORY_PROFILER_ESTIMATE_CUDAGRAPHS'] = '1'
os.environ['VLLM_ENGINE_READY_TIMEOUT_S'] = '1200'
os.environ['VLLM_NO_USAGE_STATS'] = '1'
os.environ['OTEL_SDK_DISABLED'] = 'true'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['PYTHONDONTWRITEBYTECODE'] = '1'

_CANDS = [
    Path('/kaggle/input/gemma-4-developer-agent-wheelhouse'),
    Path('/kaggle/input/datasets/metric/gemma-4-developer-agent-wheelhouse')
]
WHEELHOUSE_DIR = next((p for p in _CANDS if p.exists() and list(p.glob('*.whl'))), None)
if WHEELHOUSE_DIR is None:
    _found = sorted(Path('/kaggle/input').rglob('swegemma*.whl'))
    assert _found, 'no swegemma wheel anywhere under /kaggle/input'
    WHEELHOUSE_DIR = _found[0].parent
print('USING WHEELHOUSE:', WHEELHOUSE_DIR)

# Remove broken cutlass .pth hooks if present
for pth_pattern in (
    '/usr/local/lib/python*/dist-packages/*cutlass*.pth',
    '/usr/local/lib/python*/site-packages/*cutlass*.pth',
):
    for pth in glob.glob(pth_pattern):
        try:
            os.unlink(pth)
        except OSError as hook_error:
            # SPECIFIC, DELIBERATE TOLERANCE. The only OSError conditions
            # reachable here are FileNotFoundError (another glob match or a
            # previous cell already removed it -- success either way) and
            # PermissionError on a read-only dist-packages layer. In the second
            # case the hook stays, which degrades to the same behaviour the
            # pre-existing code tolerated, and the cell continues. This is NOT
            # a swallow-all: the handler names OSError, does not catch
            # Exception/BaseException, and logs what it did not remove so the
            # condition is visible in the log rather than invisible.
            print(f'WARNING: could not remove cutlass .pth hook {pth}: {hook_error!r}')

# Restore PEP 440 '+cu128' wheel filenames stripped by Kaggle dataset uploads
tmp_whl = Path('/tmp/wheelhouse')
tmp_whl.mkdir(parents=True, exist_ok=True)
for w in WHEELHOUSE_DIR.glob('*.whl'):
    if 'cutlass' in w.name.lower():
        continue
    target_name = (
        w.name.replace('cu128', '+cu128')
        if ('cu128' in w.name and '+' not in w.name)
        else w.name
    )
    target = tmp_whl / target_name
    if not target.exists():
        os.symlink(w, target)

wheels = sorted(str(w) for w in tmp_whl.glob('*.whl'))
print(f'Installing {len(wheels)} wheels from {WHEELHOUSE_DIR}...')
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', '--force-reinstall', *wheels],
    check=True,
)
importlib.invalidate_caches()
print('Wheelhouse installation complete.')


In [ ]:
import base64
import os
import shutil
import zipfile
from pathlib import Path
from swegemma.models import load_tasks

DATA_DIR = Path('/kaggle/input/competitions/gemma-4-developer-agent')
WORKING_DIR = Path('/kaggle/working')
WORKING_DIR.mkdir(parents=True, exist_ok=True)

AGENT_DIR = WORKING_DIR / 'my_submission'
if AGENT_DIR.exists():
    shutil.rmtree(AGENT_DIR)
AGENT_DIR.mkdir(parents=True, exist_ok=True)

# Unpack embedded verified agent bundle
B64_SUBMISSION = "..."
zip_archive_path = WORKING_DIR / 'submission.zip'
zip_archive_path.write_bytes(base64.b64decode(B64_SUBMISSION))

with zipfile.ZipFile(zip_archive_path, 'r') as zf:
    zf.extractall(AGENT_DIR)

# Purge any stray bytecode (.pyc, __pycache__)
for root, dirs, files in os.walk(AGENT_DIR):
    for f in files:
        if f.endswith('.pyc') or f == '.DS_Store':
            os.remove(os.path.join(root, f))
    for d in list(dirs):
        if d == '__pycache__':
            shutil.rmtree(os.path.join(root, d))

print('NO ADAPTER DECLARED: serving pure base weights (decision-adapter-optional).')

TASKS_PATH = DATA_DIR / 'tasks.jsonl'
tasks = load_tasks(TASKS_PATH)
GRAPH_DIR = str(DATA_DIR / 'graphs')
EMBEDDINGS_DIR = str(DATA_DIR / 'embeddings')
print(f'Data: {DATA_DIR} | Agent: {AGENT_DIR} | tasks: {len(tasks)}')

print('Unpacked submission files:')
for p in sorted(AGENT_DIR.rglob('*')):
    if p.is_file():
        print(f'  - {p.relative_to(AGENT_DIR)} ({p.stat().st_size} bytes)')


In [ ]:
import json
import queue
import threading
import time
import urllib.request

import litellm
import torch
from pathlib import Path
from adk_submission import VllmConfig, VllmServer, discover_adapters
from swegemma.config import ALLOWED_ADAPTER_EXTENSIONS
from swegemma.models.discovery import validate_single_declared_model

# ---------------------------------------------------------------- banner ----
# An operator reading only the log must never be misled about what is running.
print('=' * 78)
print(f'  TARGET MODEL : gemma-4-31b-it-qat-w4a16-ct')
print(f'  ADAPTER      : None '
      f'(0 bytes)')
print(f'  ADAPTER MOUNT: enabled -- vLLM will serve this LoRA on top of the base model.')
print('  ABOUT TO DO  : start vLLM on port 8000, health-probe /health and')
print('                /v1/chat/completions, and only then evaluate all tasks.')
print('  ON FAILURE   : this kernel ABORTS with a named error. It never reports a')
print('                score for a run it could not actually serve.')
print('=' * 78)

# ------------------------------------------------- litellm.drop_params ------
# This line used to be a bare attribute assignment, which fails INVISIBLY in
# two ways: (1) if the imported litellm has no `drop_params` attribute, Python
# happily creates one on the module and litellm keeps RAISING on unsupported
# params -- every inference call then fails far away from this line; (2) if the
# litellm in the wheelhouse moved the flag, it is set on an object nobody reads.
# So: set it, then read it back and fail loudly if it did not take.
litellm.drop_params = True
assert getattr(litellm, 'drop_params', None) is True, (
    'litellm.drop_params did not take effect: litellm will raise on '
    'unsupported params and every model call will fail far from this line. '
    'Abort instead of scoring 0 on a routing error.'
)
print('litellm.drop_params verified True (unsupported params will be dropped, not raised).')

# -------------------------------------------------- model path validation ---
# Used to be: `if not _MP.exists(): search ... ; _MP = _f[0] if _f else _MP`
# followed by a print of MODEL_PATH.exists(). That print was cosmetic: if the
# search found nothing, the ORIGINAL non-existent path was kept, vLLM was asked
# to load it, and the failure surfaced as an opaque serving error at run time.
# Now the path is a hard precondition and the search result is asserted.
TARGET_MODEL_NAME = 'gemma-4-31b-it-qat-w4a16-ct'
MODEL_WEIGHTS_MARKERS = ('config.json',)
_canonical = Path('/kaggle/input/models/google/gemma-4/other/gemma-4-31b-it-qat-w4a16-ct/2')
if not _canonical.is_dir():
    _found = sorted(
        p for p in Path('/kaggle/input').rglob('gemma-4-31b-it-qat-w4a16-ct')
        if p.is_dir() and (p / 'config.json').is_file()
    )
    if not _found:
        raise SystemExit(
            f'MODEL NOT FOUND: {_canonical} is absent and no directory containing '
            f'config.json was found for {TARGET_MODEL_NAME!r} under /kaggle/input.\n'
            '  Attach the google/gemma-4 model to the kernel and re-run. Serving '
            'without it cannot produce a valid score.'
        )
    print('model fallback search:', [str(p) for p in _found[:3]])
    _canonical = _found[0]

MODEL_PATH = _canonical
assert MODEL_PATH.is_dir(), f'MODEL PATH NOT A DIRECTORY: {MODEL_PATH}'
for _marker in MODEL_WEIGHTS_MARKERS:
    assert (MODEL_PATH / _marker).is_file(), (
        f'MODEL WEIGHTS INCOMPLETE: {MODEL_PATH / _marker} is missing. '
        'vLLM cannot load this model; abort rather than serve nothing.'
    )
print('USING MODEL:', MODEL_PATH, '| exists:', MODEL_PATH.exists(), '| config.json: OK')

INFERENCE_API_KEY = 'EMPTY'

declared_model = validate_single_declared_model(AGENT_DIR)
assert declared_model == TARGET_MODEL_NAME, (
    f'MODEL MISMATCH: agent.yaml declares {declared_model!r} but this kernel '
    f'serves {TARGET_MODEL_NAME!r}. The agent would be scored against weights '
    'it was not built for. Aborting.'
)
print('agent.yaml model matches served model:', declared_model)

adapters = []
print('NO ADAPTERS CONFIGURED: LoRA disabled for maximum KV cache.')
enable_lora_flag = False
max_loras_val = 0
max_lora_rank_val = 0

gpu_count = torch.cuda.device_count() if torch.cuda.is_available() else 1
tp_size = 4 if gpu_count >= 4 else (2 if gpu_count >= 2 else 1)
print(f'Detected GPUs: {gpu_count}, Tensor Parallel Size: {tp_size}')

# Configure vLLM with 0.90 memory utilization (official starter recommendation)
# to guarantee sufficient KV cache memory blocks for 32,768 context length
vllm_cfg = VllmConfig(
    model=str(MODEL_PATH),
    port=8000,
    host='127.0.0.1',
    tool_call_parser='gemma4',
    reasoning_parser='gemma4',
    max_model_len=32768,
    dtype='bfloat16' if (torch.cuda.is_available() and torch.cuda.is_bf16_supported()) else 'auto',
    gpu_memory_utilization=0.90,
    enable_auto_tool_choice=True,
    enable_lora=enable_lora_flag,
    max_loras=max_loras_val,
    max_lora_rank=max_lora_rank_val,
    tensor_parallel_size=tp_size,
    startup_timeout=60 * 20,
)
# Pure base weights: LoRA disabled
server_instance = VllmServer(vllm_cfg, adapter_manifest=adapters)
server_instance.start()
print(f'vLLM server started on {server_instance.base_url} (tp={tp_size})')

models = server_instance.create_model_registry(
    aliases=[declared_model, TARGET_MODEL_NAME],
    model_prefix='openai/',
    api_key=INFERENCE_API_KEY,
)
print('Model registry created successfully.')

# ------------------------------------------------------- run-health gate ----
# ADAPTED from the known-good prior art at
# docs/gemma-and-the-shape-of-doubt.md:2818 (SAMPLING_HEALTH_RUNNER), whose
# docstring is: 'Hash-pinned V14 evaluation with a separate inference health
# gate per task. The gate runs before the task timer and never enters agent
# history. It stops the campaign on infrastructure failure; it does not retry
# or change a patch.'
#
# Reused VERBATIM in shape: the /health + /v1/chat/completions '2+2 -> 4'
# probe, the daemon-thread absolute wall deadline, and EndpointUnhealthy as a
# BaseException subclass so the campaign abort escapes ordinary per-task
# `except Exception` handling. Deliberately NOT reused: install_health_gate(),
# which monkeypatches a V14-specific runner module's progress() and reads a
# V14-specific argparse Namespace (args.run_root, args.port) that does not
# exist here. Those two lines are the only V14-bound parts of the prior art.
#
# Purpose: a kernel that builds and serves a DEAD endpoint currently looks
# identical to a kernel serving a weak model. The last real run scored 0/129
# with 77/95 traces at zero completion tokens -- exactly this confusion. This
# gate makes dead-serving loud.
class EndpointUnhealthy(BaseException):
    """Campaign abort, deliberately outside ordinary per-task error handling."""


def _probe_endpoint(base, *, timeout=20):
    started = time.monotonic()
    with urllib.request.urlopen(base + '/health', timeout=timeout) as response:
        if response.status != 200:
            raise RuntimeError('Health endpoint did not return 200')
    payload = {'model': TARGET_MODEL_NAME, 'temperature': 0, 'max_tokens': 8,
               'chat_template_kwargs': {'enable_thinking': False},
               'messages': [{'role': 'user', 'content': 'What is 2+2? Reply with only the integer.'}]}
    request = urllib.request.Request(base + '/v1/chat/completions',
        data=json.dumps(payload).encode(), headers={'Content-Type': 'application/json'})
    with urllib.request.urlopen(request, timeout=timeout) as response:
        data = response.read(65537)
        if len(data) > 65536:
            raise RuntimeError('Probe response exceeded 64 KiB')
    document = json.loads(data)
    if (document['choices'][0]['message'].get('content') or '').strip() != '4':
        raise RuntimeError('Inference probe did not return 4')
    return {'status': 'PASS', 'elapsed_seconds': round(time.monotonic() - started, 3),
            'usage': document.get('usage'), 'response_sha256': __import__('hashlib').sha256(data).hexdigest()}


def probe_endpoint(base, *, timeout=20, wall_timeout=40):
    # Socket timeouts alone do not bound a slow-drip response. A daemon worker
    # gives the gate one absolute deadline; a missed deadline aborts the run.
    result = queue.Queue(maxsize=1)

    def worker():
        try:
            result.put((True, _probe_endpoint(base, timeout=timeout)))
        except BaseException as error:
            result.put((False, error))

    thread = threading.Thread(target=worker, daemon=True)
    thread.start()
    try:
        passed, value = result.get(timeout=wall_timeout)
    except queue.Empty as error:
        raise TimeoutError('Inference health gate exceeded its absolute wall deadline') from error
    if not passed:
        raise value
    return value


INFERENCE_BASE_URL = 'http://127.0.0.1:8000'
HEALTH_LOG_PATH = WORKING_DIR / 'endpoint_health.jsonl'


def require_healthy_endpoint(context):
    """Probe the endpoint or abort the whole campaign. Never enters agent history."""
    row = {'context': context, 'kind': 'VALIDATION_ONLY_INFERENCE_PROBE',
           'enters_agent_history': False}
    try:
        row.update(probe_endpoint(INFERENCE_BASE_URL))
    except Exception as error:
        row.update(status='HOLD', error=repr(error))
        with HEALTH_LOG_PATH.open('a') as stream:
            stream.write(json.dumps(row, sort_keys=True) + '\n')
        raise EndpointUnhealthy(
            'INFERENCE ENDPOINT DEAD before ' + context + ': ' + repr(error) + '\n'
            '  The kernel built and vLLM started, but the model is not being '
            'served. Every task below would score 0 for an infrastructure '
            'reason. STOP: read the vLLM log above for the load error, fix it, '
            'and re-run. This run is void.'
        ) from error
    with HEALTH_LOG_PATH.open('a') as stream:
        stream.write(json.dumps(row, sort_keys=True) + '\n')
    print(f'  endpoint health ({context}): {row["status"]} '
          f'in {row["elapsed_seconds"]}s usage={row.get("usage")}', flush=True)
    return row


require_healthy_endpoint('pre_campaign')
print('Pre-campaign endpoint health: PASS. The campaign may proceed.')


In [ ]:
import asyncio
import concurrent.futures
import json
import os
import shutil
from pathlib import Path
import pandas as pd
from google.adk.agents.context_cache_config import ContextCacheConfig
from google.adk.apps._configs import EventsCompactionConfig
from swegemma.config import EvalConfig, build_submission_limits
from swegemma.evaluate import Evaluator

def run_sync(coro_or_fn, *args, **kwargs):
    fn = (lambda: coro_or_fn(*args, **kwargs)) if callable(coro_or_fn) else (lambda: coro_or_fn)
    try:
        loop = asyncio.get_running_loop()
    except RuntimeError:
        loop = None
    if loop is not None and loop.is_running():
        with concurrent.futures.ThreadPoolExecutor(max_workers=1) as pool:
            return pool.submit(lambda: asyncio.run(fn())).result()
    return asyncio.run(fn())

# Full 129-task cloud evaluation on Kaggle 4x L4 GPUs
limits, gen_constraints = build_submission_limits()
RESULTS_DIR = WORKING_DIR / 'results'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
JSONL_PATH = RESULTS_DIR / 'task_results.jsonl'

eval_config = EvalConfig(
    tasks_path=TASKS_PATH,
    snapshots_dir=DATA_DIR / 'snapshots',
    results_dir=RESULTS_DIR,
    submission_dir=AGENT_DIR,
    models=models,
    sandbox='subprocess',
    timeout_seconds=60,
    max_time_minutes=4.5,
    max_tool_calls=40,
    max_turns=100,
    limits=limits,
    generation_constraints=gen_constraints,
    adapter_manifest=adapters,
    context_cache_config=ContextCacheConfig(min_tokens=2048, ttl_seconds=1800, cache_intervals=10),
    events_compaction_config=EventsCompactionConfig(
        compaction_interval=15, overlap_size=2,
        token_threshold=14336, event_retention_size=5),
    graph_dir=GRAPH_DIR,
    embeddings_dir=EMBEDDINGS_DIR,
    wheels_dir=DATA_DIR / 'wheels',
    verbose=False,
)
evaluator = Evaluator(eval_config)

resolved_count = 0
total_evaluated = 0
zero_token_tasks = []
predictions = []

print(f"Starting evaluation across all {len(tasks)} tasks on Kaggle 4x L4 GPUs...", flush=True)
print(f"Model under test: {TARGET_MODEL_NAME} | adapter: {ADAPTER_NAME}", flush=True)

# Degenerate-run detector. The previous real run scored 0/129 with 77/95 traces
# at ZERO completion tokens -- an infrastructure failure that looked exactly like
# model weakness. This threshold makes that class of run impossible to mistake
# for a genuine weak-model result. See also require_healthy_endpoint() in the
# serving cell, which catches the same failure earlier and more precisely.
DEGENERATE_PATCH_CHARS = 200
DEGENERATE_FRACTION = 0.9

for idx, task in enumerate(tasks, start=1):
    print(f"[{idx}/{len(tasks)}] Evaluating {task.instance_id} ({task.repo})...", flush=True)

    # Run-health gate before the task timer, per the prior art
    # (docs/gemma-and-the-shape-of-doubt.md:2818): 'The gate runs before the task
    # timer and never enters agent history. It stops the campaign on
    # infrastructure failure; it does not retry or change a patch.'
    # EndpointUnhealthy subclasses BaseException, so it deliberately escapes the
    # `except Exception` below and aborts the campaign.
    require_healthy_endpoint(f'pre_task:{task.instance_id}')

    try:
        result = run_sync(evaluator.evaluate_task, task=task, task_index=idx, total_tasks=len(tasks))
        is_resolved = bool(result.resolved)
        if is_resolved:
            resolved_count += 1
        total_evaluated += 1

        patch_text = result.agent_patch or ""
        record = {
            "task_id": task.instance_id,
            "repo": task.repo,
            "resolved": is_resolved,
            "test_exit_code": result.test_exit_code,
            "tool_calls": result.tool_calls,
            "duration_seconds": round(result.duration_seconds, 2),
            "patch_length": len(patch_text),
            "patch": patch_text,
        }
        predictions.append({"id": task.instance_id, "prediction": patch_text})

        if len(patch_text) < DEGENERATE_PATCH_CHARS:
            zero_token_tasks.append(task.instance_id)

        # Guardrail 1: Incremental flush to JSONL after every task
        with open(JSONL_PATH, "a", encoding="utf-8") as jf:
            jf.write(json.dumps(record) + "\n")

        print(
            f"  -> resolved={is_resolved} (Running Score: {resolved_count}/{total_evaluated} = {resolved_count/total_evaluated*100:.1f}%), "
            f"exit={result.test_exit_code}, patch={record['patch_length']}ch, calls={result.tool_calls}, {result.duration_seconds:.1f}s",
            flush=True
        )
    except EndpointUnhealthy:
        # Infrastructure failure. Re-raise: the campaign is void, and continuing
        # would produce a 0-score that is indistinguishable from model weakness.
        raise
    except Exception as error:
        # FAIL LOUD. This handler used to print the error, append an empty
        # prediction, and continue to the next task -- so a run in which every
        # task raised produced a clean "EVALUATION COMPLETE / Final Score:
        # 0.00%" line, indistinguishable from a model that is simply weak. That
        # is precisely the confusion this gate system exists to eliminate, and
        # it is what burned the last quota. The partial results already written
        # to task_results.jsonl are preserved for diagnosis, then the campaign
        # aborts with a named error.
        with open(JSONL_PATH, "a", encoding="utf-8") as jf:
            jf.write(json.dumps({
                "task_id": task.instance_id,
                "repo": task.repo,
                "resolved": False,
                "error": repr(error),
                "aborted": True,
            }) + "\n")
        raise SystemExit(
            f'\nEVALUATION ABORTED at task {idx}/{len(tasks)} ({task.instance_id}): {error!r}\n'
            '  An unexpected exception is NOT a task-level result; it is a bug or '
            'an infrastructure fault. Continuing would emit an empty prediction '
            'and finish with a 0-score that cannot be distinguished from model '
            f'weakness. Partial per-task results are preserved at {JSONL_PATH}. '
            'Fix the cause and re-run.'
        ) from error
    finally:
        # Guardrail 2: Clean up temporary sandbox venvs to prevent disk overflow
        for sb_dir in Path("/tmp").glob("swegemma_sandbox_*"):
            try:
                shutil.rmtree(sb_dir, ignore_errors=True)
            except OSError as cleanup_error:
                # SPECIFIC, DELIBERATE TOLERANCE. This is best-effort disk
                # hygiene running in a finally block: rmtree(ignore_errors=True)
                # already suppresses per-file errors, so the only escape here is
                # an OSError on the directory entry itself. Raising from a
                # finally would MASK the real task outcome, which is the one
                # thing this block must not do. The error is named and printed
                # so a disk-overflow failure is still visible in the log.
                print(f"WARNING: sandbox cleanup failed for {sb_dir}: {cleanup_error!r}", flush=True)

# Post-campaign degenerate-run assertion: a run in which almost every task
# produced no patch is an infra failure, not a model result. Refuse to print a
# headline score for it.
if total_evaluated:
    degenerate_fraction = len(zero_token_tasks) / total_evaluated
    print(
        f"\nDegenerate-run check: {len(zero_token_tasks)}/{total_evaluated} tasks "
        f"produced < {DEGENERATE_PATCH_CHARS} patch chars ({degenerate_fraction:.1%}).",
        flush=True,
    )
    if degenerate_fraction >= DEGENERATE_FRACTION:
        raise SystemExit(
            f'\nEVALUATION ABORTED: degenerate run -- {degenerate_fraction:.1%} of '
            f'{total_evaluated} tasks produced essentially no patch '
            f'(first few: {zero_token_tasks[:5]}).\n'
            '  This is the signature of the previous 0/129 run (77/95 traces at '
            'zero completion tokens): the model is not being served, or every '
            'generation is empty. It is NOT evidence of model weakness. Do not '
            f'read the score as one. Per-task results: {JSONL_PATH}; endpoint '
            f'health log: {HEALTH_LOG_PATH}.'
        )

print(f"\n=================== EVALUATION COMPLETE ===================", flush=True)
print(f"Final Score: {resolved_count}/{total_evaluated} ({resolved_count/max(1, total_evaluated)*100:.2f}%)", flush=True)
submission_df = pd.DataFrame(predictions, columns=['id', 'prediction'])
submission_df.to_csv(RESULTS_DIR / 'predictions.csv', index=False)
print(f"Saved predictions.csv ({len(submission_df)} rows)", flush=True)


In [ ]:
import zipfile
from swegemma.config import ALLOWED_SUBMISSION_EXTENSIONS, MAX_SUBMISSION_SIZE_BYTES

# Finalize submission.zip in /kaggle/working/
zip_base = WORKING_DIR / 'submission'
zip_path = Path(shutil.make_archive(str(zip_base), 'zip', root_dir=AGENT_DIR))

with zipfile.ZipFile(zip_path, 'r') as zf:
    infos = zf.infolist()
    total_size = sum(i.file_size for i in infos)
    assert total_size <= MAX_SUBMISSION_SIZE_BYTES, f'Archive exceeds 3 GiB: {total_size}'
    for info in infos:
        if not info.is_dir():
            ext = Path(info.filename).suffix.lower()
            assert ext in ALLOWED_SUBMISSION_EXTENSIONS, f'Disallowed ext: {info.filename}'

print(f'SUCCESS: Verified submission archive ready at {zip_path}')
print(f'Archive Size: {zip_path.stat().st_size / (1024 * 1024):.2f} MiB (limit: 3072 MiB)')
